# API preparation

Export small lookup tables that the strategy API needs into `src/data/`.
`data/` is not committed to git, so the API cannot read from there.

In [1]:
import pandas as pd
from pathlib import Path

src_path = Path("../data/processed/pit_loss.parquet")
pit_loss = pd.read_parquet(src_path)

print(pit_loss.shape)
pit_loss.head()

(24, 3)


,Event,PitLoss,Stops
0,Monaco Grand Prix,19.084,41
1,Canadian Grand Prix,19.747,45
2,Saudi Arabian Grand Prix,20.203,21
3,Belgian Grand Prix,20.262,45
4,Australian Grand Prix,20.462,30


In [2]:
assert pit_loss["Event"].is_unique
assert pit_loss[["Event", "PitLoss"]].notna().all().all()
assert pit_loss["PitLoss"].between(10, 45).all()

print("OK:", len(pit_loss), "events")

OK: 24 events


In [3]:
out_dir = Path("../src/data")
out_dir.mkdir(exist_ok=True)

out_path = out_dir / "pit_loss.csv"
pit_loss.round(2).to_csv(out_path, index=False)

print("saved:", out_path.resolve())

saved: C:\Users\vova1\Documents\vscodefiles\resume projects\f1 project\src\data\pit_loss.csv


In [4]:
check = pd.read_csv(out_path)

print(check.shape)
print(check.dtypes)
check.head()

(24, 3)
Event       object
PitLoss    float64
Stops        int64
dtype: object


,Event,PitLoss,Stops
0,Monaco Grand Prix,19.08,41
1,Canadian Grand Prix,19.75,45
2,Saudi Arabian Grand Prix,20.20,21
3,Belgian Grand Prix,20.26,45
4,Australian Grand Prix,20.46,30


## Check src/strategy.py

In [5]:
import sys
sys.path.append("..")

from src.strategy import optimal_pit_lap, pit_window, load_pit_loss, plan_one_stop

In [ ]:
print(optimal_pit_lap(53))   
print(optimal_pit_lap(50))   

25.322962962962958
33.23111111111111
23.9274074074074


In [7]:
print(optimal_pit_lap(16))   # минимальная гонка → 8 (сработает граница)

try:
    optimal_pit_lap(10)
except ValueError as e:
    print("ValueError:", e)

8
ValueError: race_laps must be at least 16


In [8]:
print(pit_window(53))        # (20, 31)
print(pit_window(53, 2.0))   # шире, примерно (18, 33)
print(pit_window(20))        # (8, 12) — обрезано границами

(20, 31)
(18, 33)
(8, 12)


In [9]:
pit_loss = load_pit_loss()

print(len(pit_loss))                     # 24
print(pit_loss["Italian Grand Prix"])    # 26.1
print(pit_loss["Monaco Grand Prix"])     # 19.08
print(type(pit_loss["Monaco Grand Prix"]))

24
26.06
19.08
<class 'float'>


In [10]:
plan_one_stop("Italian Grand Prix", 53, pit_loss)

{'event': 'Italian Grand Prix',
 'race_laps': 53,
 'strategy': 'MEDIUM -> HARD',
 'optimal_pit_lap': 25,
 'window_start': 20,
 'window_end': 31,
 'pit_loss_sec': 26.06}

In [11]:
try:
    plan_one_stop("Monza", 53, pit_loss)
except KeyError as e:
    print("KeyError:", e)

KeyError: 'Unknown event: Monza'
